# jetfighter-sim : Project overview

**A fighter-jet flight simulator (F-16 parameters), built from the physics up and validated
layer by layer.**

This repository is the physics half of a two-repository project. The package it provides,
`jetsim`, is used as a dependency by [**jetfighter-rl**](https://github.com/elouansalaun/jetfighter-rl), which trains
reinforcement-learning agents to fly maneuvers on top of it (phases 7–8).

Each notebook is short, runs on a laptop in under a minute, and mixes three things:

* **the idea** : what the phase is for and why it was designed this way;
* **the equations** : only the ones that matter, with the notation used in the code;
* **small runnable examples** : calling the real `jetsim` code, with plots.

| # | Notebook | What it shows |
|---|---|---|
| 0 | [`00_project_overview`](00_project_overview.ipynb) | Goal, architecture, conventions, validation at a glance |
| 1 | [`01_physics_foundations`](01_physics_foundations.ipynb) | Reference frames, quaternions, standard atmosphere, RK4 integrator |
| 2 | [`02_point_mass_3dof`](02_point_mass_3dof.ipynb) | A fast "point-mass" aircraft: forces, trim, flight envelope, a loop |
| 3 | [`03_rigid_body_6dof`](03_rigid_body_6dof.ipynb) | The full rigid-body F-16 with NASA wind-tunnel tables, stability modes |
| 4 | [`04_instruments_envelope`](04_instruments_envelope.ipynb) | Cockpit instruments, noisy sensors, energy, "you crashed" detection |
| 5 | [`05_visualization`](05_visualization.ipynb) | Flight recording, deterministic replay, plots, Tacview 3D export |
| 6 | [`06_classical_control`](06_classical_control.ipynb) | PID, fly-by-wire, autopilot, LQR : the baseline an RL agent must beat |
| 7–8 | [jetfighter-rl](https://github.com/elouansalaun/jetfighter-rl/tree/main/notebooks) | Gymnasium environment, PPO agents, curriculum, imitation learning |

## 1. The goal

The end goal of the overall project is an agent that **flies a fighter jet and is capable of maneuvering**. That is a hard problem, so it is reached through a ladder of easier ones:

1. a trustworthy flight simulator (phases 1–5) : **this repository**;
2. a classical autopilot as a reference (phase 6) : **this repository**;
3. a reinforcement-learning environment (phase 7) : [jetfighter-rl](https://github.com/elouansalaun/jetfighter-rl);
4. a curriculum of maneuvers learned by RL ; stabilize, turn, navigate, loop (phase 8) :
   [jetfighter-rl](https://github.com/elouansalaun/jetfighter-rl);
5. then a missile model and evasion training (phases 9–10, not started yet).



## 2. Architecture

```
                  ┌──────────────────────────────────────────────────┐
                  │  RL agent (PPO, SAC, imitation)   phase 8        │
jetfighter-rl     │  Gymnasium env: tasks, rewards    phase 7        │
                  └─────────┬────────────────────────────▲───────────┘
                    command │                            │ observation,
                            │                            │ reward, done
                  ┌─────────▼────────────────────────────┴───────────┐
                  │  control/   fly-by-wire, autopilot  phase 6      │
jetsim            │  aircraft/  3-DOF & 6-DOF dynamics  phases 2-3   │  ◄ this repo
                  │             instruments, sensors,                │
                  │             flight envelope         phase 4      │
                  │  core/      frames, ISA, RK4        phase 1      │
                  │  viz/       recording, Tacview      phase 5      │
                  └──────────────────────────────────────────────────┘
```

`jetsim` knows nothing about reinforcement learning: it exposes aircraft models, instruments,
controllers and visualization tools through plain Python interfaces, and the RL repository
wraps them into a Gymnasium environment.

Two aircraft models share the same interfaces, so everything above them (autopilot,
environment, agents) works with either:

* **3-DOF point mass** : very fast, used to prototype control laws and RL tasks;
* **6-DOF rigid body** : realistic, driven by its actual control surfaces, using the
  F-16 aerodynamic tables published by NASA (Stevens & Lewis).

## 3. Key design decisions

The roadmap (`project_roadmap.md`) keeps a decision log. The entries that shaped the simulator:

| Decision | Why |
|---|---|
| **F-16** as the reference aircraft (not F-22) | Its aerodynamic data is public (NASA TP-1538), so the 6-DOF model uses real wind-tunnel tables |
| **Quaternions** for attitude, Euler angles only for display | Euler angles are singular at ±90° pitch : exactly where a loop goes |
| Default CG at **x_cg = 0.30** (stable), 0.35 available | A naturally stable aircraft is simpler to start with; the relaxed-stability 0.35 (like the real F-16) is stabilized by the fly-by-wire of phase 6 |
| Fly-by-wire load-factor command **rate-limited to 20 g/s**, with pitch-rate anticipation | Abrupt command reversals ; typical of an RL agent, overshot −4 g (up to −5 g) |
| Aircraft data **shipped inside the package** (`jetsim/data/`) | `jetsim` works the same from a `pip install` as from a source checkout, so other projects can depend on it |

Two more entries were found downstream, by the RL agents (see notebooks 1 and 2): observing
Euler angles makes an agent "think it is inverted" at the top of a loop, and a turn-rate
reward can be gamed by a descending spiral.

## 4. Conventions

Consistency is enforced across the whole code base, in both repositories:

| Topic | Convention |
|---|---|
| Units | **SI everywhere** internally; conversions only at the boundaries |
| Angles | **Radians** internally; degrees only for display and config files |
| Inertial frame | **NED** (North-East-Down), flat Earth; altitude $h = -z$ |
| Body frame | **FRD**: $x$ out the nose, $y$ out the right wing, $z$ down |
| Attitude | Unit quaternion $q = [q_0, q_1, q_2, q_3]$ (scalar first), $v_{ned} = C_{nb}(q)\,v_{body}$ |
| Time step | Physics at **100 Hz** with RK4; controllers and agents decide at 10–50 Hz |
| Matrix names | $C_{ab}$ maps a vector from frame $b$ to frame $a$ |

## 5. The package in numbers

Let's look at the actual package. Each sub-package maps to one or more phases.

In [16]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from jetsim.viz.style import apply_style

apply_style()
plt.rcParams["figure.dpi"] = 110
ROOT = Path.cwd().parent  # notebooks/ -> repository root

PHASES = {
    "core": "1 · frames, atmosphere, integrators",
    "aircraft": "2-4 · 3-DOF / 6-DOF models, instruments",
    "viz": "5 · recording, plots, Tacview",
    "control": "6 · PID, fly-by-wire, autopilot, LQR",
}
rows = []
for pkg, phase in PHASES.items():
    files = sorted((ROOT / "src/jetsim" / pkg).rglob("*.py"))
    lines = sum(len(f.read_text(encoding="utf-8").splitlines()) for f in files)
    rows.append((pkg, phase, len(files), lines))
print(f"{'package':<10}{'phase · content':<44}{'files':>6}{'lines':>8}")
for pkg, phase, n, lines in rows:
    print(f"{pkg:<10}{phase:<44}{n:>6}{lines:>8}")
print(f"{'total':<54}{sum(r[2] for r in rows):>6}{sum(r[3] for r in rows):>8}")

package   phase · content                              files   lines
core      1 · frames, atmosphere, integrators              5     790
aircraft  2-4 · 3-DOF / 6-DOF models, instruments         12    2207
viz       5 · recording, plots, Tacview                    6     915
control   6 · PID, fly-by-wire, autopilot, LQR             6     568
total                                                     29    4480


### Tooling

* `pyproject.toml` + [uv](https://docs.astral.sh/uv/) for dependencies, with optional extras
  (`[viz]` for pygame manual flight, `[notebooks]` for JupyterLab);
* `ruff` for lint and formatting, run by `pre-commit` hooks;
* a GitHub Actions workflow that lints and runs the tests on Python 3.10 and 3.12;
* YAML files for the aircraft and the sensors, shipped inside the package.

The test suite is where the physics validation lives (analytic solutions, published values,
conservation laws). Running it:

In [17]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout.strip().splitlines()[-1])

596 passed in 10.09s


## 6. Validation at a glance

The 3-DOF model is checked against published F-16 performance (`scripts/phase2_performance.py`):

| Quantity | Model | Public reference |
|---|---|---|
| Max Mach at sea level | 1.18 | ≈ 1.2 |
| Max Mach at 11 km | 2.08 | ≈ 2.0 |
| Max climb rate at sea level | 275 m/s | ≈ 250 m/s |
| Ceiling (0.5 m/s climb) | 17.7 km | 15.2 km (operational) |
| Sustained turn rate, 3 km, M 0.8 | 18.3 °/s | ≈ 18–20 °/s |

The 6-DOF model reproduces the classic aircraft modes (short period, phugoid, Dutch roll,
roll subsidence, spiral), notebook 3 computes them from the linearized model. Below, the
6-DOF F-16 is trimmed at four flight conditions: the throttle, elevator and angle of attack
needed for steady level flight.

In [18]:
import time

from jetsim.aircraft import dynamics_6dof as d6
from jetsim.aircraft.instruments import read_instruments
from jetsim.aircraft.params import load_aircraft

f16 = d6.F16SixDof(load_aircraft("f16"))
print(f"{'flight condition':<20}{'throttle':>9}{'elevator':>10}{'alpha':>8}{'Mach':>7}")
for h, v in ((0, 153), (3000, 200), (6000, 250), (9000, 250)):
    x, u = f16.trim(h, v)
    ins = read_instruments(f16, x)
    print(
        f"{h:>5} m {v:>4} m/s      {u[d6.THROTTLE]:>7.2f}"
        f"{np.degrees(u[d6.ELEVATOR]):>9.1f}°{np.degrees(ins.alpha):>7.1f}°{ins.mach:>7.2f}"
    )

x, u = f16.trim(3000, 200)
start, n = time.perf_counter(), 2000
for _ in range(n):
    x = f16.step(x, u)
rate = n / (time.perf_counter() - start)
print(f"\n6-DOF speed: {rate:,.0f} physics steps/s = {rate * 0.01:.0f}x real time (RK4, 100 Hz)")

flight condition     throttle  elevator   alpha   Mach
    0 m  153 m/s         0.15     -1.9°    2.3°   0.45
 3000 m  200 m/s         0.22     -1.7°    1.5°   0.61
 6000 m  250 m/s         0.36     -1.7°    1.1°   0.79
 9000 m  250 m/s         0.34     -1.9°    2.2°   0.82

6-DOF speed: 7,411 physics steps/s = 74x real time (RK4, 100 Hz)


## 7. Using `jetsim` from another project

`jetsim` is a regular Python package, installable straight from GitHub:

```bash
pip install "jetsim @ git+https://github.com/elouansalaun/jetfighter-sim.git@v0.1.0"
```

This is exactly how [jetfighter-rl](https://github.com/elouansalaun/jetfighter-rl) depends on it (see its `pyproject.toml`).

## 8. How to run these notebooks

```bash
uv venv && uv pip install -e ".[dev,notebooks]"
uv run jupyter lab notebooks/
```